In [1]:
import os
import sys
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import yaml
from tqdm import tqdm

# Add parent directory to path
# sys.path.insert(0, str(Path(__file__).parent.parent))
# sys.path.insert(0, str(Path(__file__).parent.parent / "dataset" / "eth"))

os.environ["MOMENTUM_ENABLED"] = "1"

sys.path.insert(0, "/home/rikhat/human_global_motion/sam-3d-body")

from sam_3d_body.build_models import load_sam_3d_body
from sam_3d_body.utils.config import get_config
from dataset_utils import prepare_training_batch

/home/rikhat/human_global_motion/sam-3d-body/sam_3d_body/models/heads/mhr_head.py:29: UserWarning: Momentum is not enabled
  warnings.warn("Momentum is not enabled")
/home/rikhat/human_global_motion/sam-3d-body/sam_3d_body/models/heads/mhr_head.py:33: UserWarning: Momentum is not enabled
  warnings.warn("Momentum is not enabled")


In [2]:
cfg = get_config("config.yaml")
    
# Load base model
print("Loading SAM-3D-Body model...")
model, model_cfg = load_sam_3d_body(
    checkpoint_path=cfg.MODEL.CHECKPOINT_PATH,
    device="cuda",
    mhr_path=cfg.MODEL.MHR_MODEL_PATH
)

# Load trained contact head
print(f"Loading trained contact head from {"output/contact_head_eth/best_model.pth"}")
checkpoint = torch.load("output/contact_head_eth/best_model.pth", map_location="cuda")
if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
else:
    model.load_state_dict(checkpoint, strict=False)

model.eval()

Loading SAM-3D-Body model...
Loading SAM 3D Body model...


Using cache found in /home/rikhat/.cache/torch/hub/facebookresearch_dinov3_main
Ignored kwargs: {'drop_path': 0.1}
The model and loaded state dict do not match exactly

missing keys in source state_dict: backbone.encoder.mask_token, head_pose.hand_pose_comps_ori, head_pose.mhr.face_expressions_model.shape_vectors, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.0.sparse_indices, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.0.sparse_weight, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.2.weight, head_pose.mhr.character_torch.skeleton.joint_translation_offsets, head_pose.mhr.character_torch.skeleton.joint_prerotations, head_pose.mhr.character_torch.skeleton.pmi, head_pose.mhr.character_torch.skeleton.joint_parents, head_pose.mhr.character_torch.mesh.rest_vertices, head_pose.mhr.character_torch.mesh.faces, head_pose.mhr.character_torch.mesh.texcoords, head_pose.mhr.character_torch.mesh.texcoord_faces, head_pose.mhr.character_torch.parameter_transform.parame

Loading trained contact head from output/contact_head_eth/best_model.pth


SAM3DBody(
  (backbone): Dinov3Backbone(
    (encoder): DinoVisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1280, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (rope_embed): RopePositionEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x SelfAttentionBlock(
          (norm1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (attn): SelfAttention(
            (qkv): LinearKMaskedBias(in_features=1280, out_features=3840, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=1280, out_features=1280, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (norm2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (mlp): SwiGLUFFN(
            (w1): Linear(in_features=1280, out_features=5120, bias=True)
            (w2): Linear(in_features=1280, out_features=5120, bias=True)
  

In [3]:
data_path = "/home/rikhat/climbing/climb_forces/data/eth"
folder_name = "2"
video_name = "07_peter_climbing"
side = "left"

config_path = f"{data_path}/{folder_name}/data/{video_name}/config.yaml"
config = yaml.safe_load(open(config_path, 'r'))

kp_path = f"{data_path}/{folder_name}/data/{video_name}/dense_kp_{side}.npy"
kp = np.load(kp_path, allow_pickle=True).item()
bbox = kp['bbox']

keypoints_path = f"{data_path}/{folder_name}/data/{video_name}/keypoints_{side}.npy"
keypoints = np.load(keypoints_path, allow_pickle=True).item()

frames_path = f"{data_path}/{folder_name}/frames/{video_name}/{side}"

frame_start = config['frame_start']
frame_end = config['frame_end']

writer = cv2.VideoWriter('output/2_07.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 30, (1920, 1080))

for frame_id in tqdm(range(frame_start, frame_end)):

    frame_path = os.path.join(frames_path, f"{frame_id:06d}.jpg")
    frame = cv2.imread(frame_path)
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    box = bbox[frame_id].astype(np.int32)

    cam_params = {
        'fx': cfg.CAMERA.fx,
        'fy': cfg.CAMERA.fy,
        'cx': cfg.CAMERA.cx,
        'cy': cfg.CAMERA.cy
    }
    
    batch = prepare_training_batch(
        [frame],
        [box],
        cam_params,
        target_size=(896, 896),  # Maximum resolution with DINOv3
        device="cuda"
    )

    with torch.no_grad():
        model._initialize_batch(batch)
        output = model.forward_step(batch, decoder_type="body")
        contact_logits = output["contact"]["contact_logits"]
        contact_probs = torch.sigmoid(contact_logits).cpu().numpy()[0]
        # contact_pred = (contact_probs > 0.5).astype(bool)

    mhr_kps = output["mhr"]["pred_keypoints_2d"].cpu().numpy()[0]

    
    for i, kp in enumerate(mhr_kps[[62, 41, 16, 19]]):
        x, y = kp
        if i in [0, 1] and contact_probs[i] > 0.75:
            cv2.circle(frame, (int(x), int(y)), 5, (255, 0, 0), -1)
        elif i in [2, 3] and contact_probs[i] > 0.8:
            cv2.circle(frame, (int(x), int(y)), 5, (255, 0, 0), -1)
        else:
            cv2.circle(frame, (int(x), int(y)), 5, (0, 255, 0), -1)
        # cv2.putText(frame, str(i), (int(x), int(y)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)  

    # print("Original order:", contact_probs)
    # plt.figure(figsize=(30, 20))
    # plt.imshow(frame)
    # plt.axis('off')
    # plt.show()
    # break

    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    writer.write(frame)

writer.release()


  0%|          | 0/401 [00:00<?, ?it/s]

100%|██████████| 401/401 [01:01<00:00,  6.53it/s]


In [22]:
output["mhr"]["pred_keypoints_2d"].shape

torch.Size([1, 70, 2])